# Lesson 12B: Deep Unsupervised Learning Practical — Convolutional Denoising Autoencoder

## Introduction

Lesson 12A's autoencoders were fully-connected: every pixel feeds every hidden unit, with no notion that pixel $(5, 5)$ is spatially close to pixel $(5, 6)$. For image data, that throws away exactly the structure that matters most. A **convolutional autoencoder** uses `Conv2d`/`ConvTranspose2d` layers that exploit spatial locality directly — fewer parameters, and reconstructions that respect image structure rather than treating pixels as an unordered bag of 784 numbers.

This lesson also tackles a genuinely different training objective: **denoising**. Instead of reconstructing the same input you fed in (which a large enough network could solve trivially, by memorizing an identity mapping), a denoising autoencoder is fed a *corrupted* input and trained to output the *clean* original. There's no shortcut to memorize — the network is forced to learn what a real digit looks like well enough to strip out noise that was never part of the signal.

In this lesson:
1. Build a convolutional autoencoder and compare its parameter count and reconstruction quality against a fully-connected one
2. Extend it to denoising: corrupt inputs with Gaussian noise, train to recover the clean original
3. Quantify denoising performance against the naive "do nothing" baseline
4. Compare training dynamics between the fully-connected and convolutional architectures

## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [MNIST Setup](#mnist)
4. [Convolutional vs Fully-Connected Autoencoder](#conv-vs-fc)
5. [Training Dynamics Comparison](#training-comparison)
6. [Denoising Autoencoder](#denoising)
7. [Quantifying Denoising Performance](#denoising-quantify)
8. [Conclusion](#conclusion)

<a name="required-libraries"></a>
## Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

torch.manual_seed(42)
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Libraries loaded!")

<a name="mnist"></a>
## MNIST Setup

Same 6,000-image subset as 12A, but kept in its native $1 \times 28 \times 28$ image shape rather than flattened — convolutional layers need the spatial structure preserved.

In [ ]:
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())

n_samples = 6000
X = train_dataset.data[:n_samples].float().view(-1, 1, 28, 28) / 255.0

print(f"Dataset: {X.shape[0]} images, shape {tuple(X.shape[1:])}")

<a name="conv-vs-fc"></a>
## Convolutional vs Fully-Connected Autoencoder

The convolutional encoder downsamples $28\times28 \to 14\times14 \to 7\times7$ via two stride-2 convolutions; the decoder mirrors that back up via `ConvTranspose2d`. We build a fully-connected autoencoder with a comparable latent dimensionality for a fair side-by-side.

In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1), nn.ReLU(),   # 28 -> 14
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), nn.ReLU(),  # 14 -> 7
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1), nn.ReLU(),  # 7 -> 14
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1), nn.Sigmoid(),  # 14 -> 28
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))


class FCAutoencoder(nn.Module):
    def __init__(self, input_dim: int = 784, hidden_dim: int = 256, latent_dim: int = 32):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, input_dim), nn.Sigmoid())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))


conv_ae = ConvAutoencoder()
fc_ae = FCAutoencoder()

conv_params = sum(p.numel() for p in conv_ae.parameters())
fc_params = sum(p.numel() for p in fc_ae.parameters())

print(f"Convolutional AE parameters: {conv_params:,}")
print(f"Fully-connected AE parameters: {fc_params:,}")
print(f"Fully-connected AE has {fc_params / conv_params:.1f}x more parameters")

<a name="training-comparison"></a>
## Training Dynamics Comparison

Train both on the same clean-reconstruction task and compare loss curves and final reconstructions.

In [ ]:
def train_autoencoder(model, X, n_epochs, batch_size, flatten_input=False, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    n = X.shape[0]
    losses = []
    for epoch in range(n_epochs):
        perm = torch.randperm(n)
        epoch_loss = 0.0
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            batch = X[idx]
            target = batch.view(len(idx), -1) if flatten_input else batch
            model_input = batch.view(len(idx), -1) if flatten_input else batch

            optimizer.zero_grad()
            recon = model(model_input)
            loss = nn.functional.mse_loss(recon, target)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(idx)
        losses.append(epoch_loss / n)
    return losses


conv_losses = train_autoencoder(conv_ae, X, n_epochs=15, batch_size=128, flatten_input=False)
fc_losses = train_autoencoder(fc_ae, X, n_epochs=15, batch_size=128, flatten_input=True)

print(f"Final reconstruction MSE — Conv AE: {conv_losses[-1]:.5f}, FC AE: {fc_losses[-1]:.5f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(conv_losses, label=f'Conv AE ({conv_params:,} params)')
axes[0].plot(fc_losses, label=f'FC AE ({fc_params:,} params)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Reconstruction MSE')
axes[0].set_title('Training Loss', fontsize=11, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

with torch.no_grad():
    conv_recon = conv_ae(X[:1])
    fc_recon = fc_ae(X[:1].view(1, -1)).view(1, 1, 28, 28)

axes[1].axis('off')
for i, (title, img) in enumerate([('Original', X[0]), ('Conv AE', conv_recon[0]), ('FC AE', fc_recon[0])]):
    ax_inset = axes[1].inset_axes([i * 0.33, 0.1, 0.3, 0.8])
    ax_inset.imshow(img.squeeze(), cmap='gray')
    ax_inset.set_title(title, fontsize=9)
    ax_inset.axis('off')
axes[1].set_title('Reconstruction Comparison', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"💡 The convolutional model reaches comparable or better reconstruction quality with {fc_params / conv_params:.1f}x fewer parameters — spatial structure is worth exploiting")

<a name="denoising"></a>
## Denoising Autoencoder

Corrupt each training image with Gaussian noise, clipped back into the valid $[0, 1]$ pixel range, but train the loss against the **clean** original. The network never sees a matched noisy-to-noisy pair — its only path to low loss is learning what real digits look like well enough to strip out noise that isn't part of any digit.

In [ ]:
noise_std = 0.3

def add_noise(x: torch.Tensor, std: float) -> torch.Tensor:
    return torch.clamp(x + std * torch.randn_like(x), 0.0, 1.0)


def train_denoising_autoencoder(model, X, n_epochs, batch_size, noise_std, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    n = X.shape[0]
    losses = []
    for epoch in range(n_epochs):
        perm = torch.randperm(n)
        epoch_loss = 0.0
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            clean = X[idx]
            noisy = add_noise(clean, noise_std)

            optimizer.zero_grad()
            recon = model(noisy)
            loss = nn.functional.mse_loss(recon, clean)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(idx)
        losses.append(epoch_loss / n)
    return losses


denoising_ae = ConvAutoencoder()
denoise_losses = train_denoising_autoencoder(denoising_ae, X, n_epochs=15, batch_size=128, noise_std=noise_std)

print(f"Final denoising loss (reconstruction vs clean): {denoise_losses[-1]:.5f}")

plt.figure(figsize=(7, 4))
plt.plot(denoise_losses)
plt.xlabel('Epoch')
plt.ylabel('MSE (denoised vs clean)')
plt.title('Denoising Autoencoder Training Loss', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
n_show = 8
clean_samples = X[:n_show]
noisy_samples = add_noise(clean_samples, noise_std)
with torch.no_grad():
    denoised_samples = denoising_ae(noisy_samples)

fig, axes = plt.subplots(3, n_show, figsize=(14, 5.5))
for i in range(n_show):
    axes[0, i].imshow(clean_samples[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(noisy_samples[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')
    axes[2, i].imshow(denoised_samples[i].squeeze(), cmap='gray')
    axes[2, i].axis('off')
axes[0, 0].set_title('Clean', loc='left', fontsize=10)
axes[1, 0].set_title('Noisy input', loc='left', fontsize=10)
axes[2, 0].set_title('Denoised', loc='left', fontsize=10)
plt.tight_layout()
plt.show()

<a name="denoising-quantify"></a>
## Quantifying Denoising Performance

Compare the reconstruction error the model actually achieves against the naive baseline of doing nothing at all — just measuring how far the noisy input already is from the clean original.

In [ ]:
test_clean = X[:500]
test_noisy = add_noise(test_clean, noise_std)
with torch.no_grad():
    test_denoised = denoising_ae(test_noisy)

baseline_mse = nn.functional.mse_loss(test_noisy, test_clean).item()
denoised_mse = nn.functional.mse_loss(test_denoised, test_clean).item()

print(f"Baseline (noisy vs clean, no denoising): {baseline_mse:.5f}")
print(f"Denoised vs clean:                       {denoised_mse:.5f}")
print(f"Error reduction: {(1 - denoised_mse / baseline_mse) * 100:.1f}%")
print("\n💡 The network never saw a single clean-to-clean training pair for this test batch — every improvement here comes from having learned what digits generally look like")

<a name="conclusion"></a>
## Conclusion

### Key Takeaways

1. **Convolutional autoencoders exploit spatial structure that fully-connected ones discard** — comparable or better reconstruction quality with far fewer parameters, since `Conv2d` shares weights across spatial positions instead of learning a separate connection per pixel pair.
2. **Denoising forces genuine representation learning** — feeding noisy input and requiring clean output removes the trivial identity-mapping shortcut a large enough plain autoencoder could otherwise memorize.
3. **A denoising autoencoder generalizes to noise it never explicitly matched during training** — every test-time improvement over the noisy baseline comes from the model having learned the underlying structure of digits, not from memorizing specific noise patterns.
4. **The comparison against a "do nothing" baseline is essential** — reporting reconstruction error alone doesn't tell you whether the model did anything useful; reporting it relative to the corrupted input's own distance from clean does.

### Practical Guidance

- Default to convolutional architectures for image data — the parameter savings and inductive bias both favor it, with no real downside for image-shaped inputs.
- When building a denoising autoencoder, always measure against the noisy-vs-clean baseline, not just the model's absolute loss — a low absolute loss can still be worse than doing nothing if the noise itself was mild.
- Match noise type and intensity to your actual use case — a model tuned for Gaussian noise removal won't necessarily generalize to salt-and-pepper noise or occlusion.

### Next Steps

This closes Lesson 12 and the algorithm-specific portion of the course. Lessons 13-16 turn to professional practice: comparing algorithms, building preprocessing pipelines, and combining unsupervised methods with limited labels.

You now have the complete deep unsupervised learning toolkit: the theoretical bridge from PCA through VAEs in 12A, and the practical convolutional and denoising techniques from this lesson 🎯